In [ ]:
# ============================================================
# PREDICTIVE MAINTENANCE — IMPROVED FULL PIPELINE
# CMAPSS FD001 Dataset
# ============================================================
# KEY FIXES vs. original:
#  1. Drop ALL near-zero variance features (op1, op2, op3 + dead sensors)
#  2. Global MinMaxScaler (fit on train only, apply to both) — no per-unit
#     normalization that causes train/test distribution mismatch
#  3. GroupKFold by engine unit — prevents temporal data leakage across folds
#  4. RUL capped at 125 (piecewise linear: early-life assumed healthy)
#  5. EWMA rolling features — captures degradation TREND, not just level
#  6. Full test-set evaluation using RUL_FD001.txt (the held-out ground truth)
#  7. RandomForest added for comparison
#  8. VIF computed on raw (not per-unit-normalized) features — meaningful results
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, GroupKFold,
                                     cross_validate, learning_curve)
from sklearn.metrics import (RocCurveDisplay, ConfusionMatrixDisplay,
                             f1_score, roc_auc_score, precision_score,
                             recall_score, classification_report)
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor

os.makedirs('models',  exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# ── Consistent visual style ──────────────────────────────────
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.alpha': 0.3})

# ============================================================
# 1. LOAD DATA
# ============================================================
col_names = (
    ['unit', 'cycle'] +
    ['op1', 'op2', 'op3'] +
    [f'sensor{i}' for i in range(1, 22)]
)

train_df = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=col_names)
test_df  = pd.read_csv('test_FD001.txt',  sep=r'\s+', header=None, names=col_names)
rul_df   = pd.read_csv('RUL_FD001.txt',   header=None, names=['true_rul'])

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"RUL entries : {len(rul_df)}  (one per test engine)")

# ============================================================
# 2. FEATURE SELECTION — drop zero / near-zero variance columns
# ============================================================
# Near-zero: op1≈5e-6, op2≈9e-8, op3=0; sensor1,5,6,10,16,18,19=0
DROP_COLS = ['op1', 'op2', 'op3',
             'sensor1', 'sensor5', 'sensor6', 'sensor10',
             'sensor16', 'sensor18', 'sensor19']

train_df.drop(columns=DROP_COLS, inplace=True)
test_df.drop(columns=DROP_COLS, inplace=True)

RAW_FEATURES = [c for c in train_df.columns
                if c.startswith('sensor') or c.startswith('op')]

print(f"\nRetained {len(RAW_FEATURES)} raw sensor features: {RAW_FEATURES}")

# ============================================================
# 3. COMPUTE RUL + BINARY LABEL  (train set)
# ============================================================
RUL_CAP   = 125   # piecewise linear: engines assumed healthy above this
THRESHOLD = 30    # binary label: 1 = within 30 cycles of failure

max_cycles       = train_df.groupby('unit')['cycle'].max().rename('max_cycle')
train_df         = train_df.join(max_cycles, on='unit')
train_df['RUL']  = (train_df['max_cycle'] - train_df['cycle']).clip(upper=RUL_CAP)
train_df['label']= (train_df['RUL'] <= THRESHOLD).astype(int)
train_df.drop(columns=['max_cycle'], inplace=True)

print(f"\nLabel distribution (train):\n{train_df['label'].value_counts().to_string()}")
print(f"Positive rate : {train_df['label'].mean():.2%}")

# ============================================================
# 4. COMPUTE RUL + BINARY LABEL  (test set using RUL_FD001.txt)
# ============================================================
# RUL_FD001.txt gives the true RUL at the LAST recorded cycle of each unit.
# We back-calculate RUL for every cycle in the test set.
rul_df['unit'] = range(1, len(rul_df) + 1)
last_cycle     = test_df.groupby('unit')['cycle'].max().rename('last_cycle')
test_df        = test_df.join(last_cycle, on='unit')
test_df        = test_df.join(rul_df.set_index('unit')['true_rul'], on='unit')
test_df['RUL'] = (test_df['true_rul'] + test_df['last_cycle'] - test_df['cycle']).clip(upper=RUL_CAP)
test_df['label'] = (test_df['RUL'] <= THRESHOLD).astype(int)
test_df.drop(columns=['last_cycle', 'true_rul'], inplace=True)

print(f"\nLabel distribution (test — full sequence):\n{test_df['label'].value_counts().to_string()}")

# ============================================================
# 5. EDA — degradation trend for a sample of engines
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
sample_sensors = ['sensor2', 'sensor3', 'sensor4', 'sensor7', 'sensor11', 'sensor12']
sample_units   = np.random.choice(train_df['unit'].unique(), 6, replace=False)

for ax, sensor in zip(axes.flatten(), sample_sensors):
    for unit in sample_units:
        sub = train_df[train_df['unit'] == unit].sort_values('cycle')
        ax.plot(sub['cycle'], sub[sensor], alpha=0.5, linewidth=0.8)
    ax.set_title(sensor, fontsize=10)
    ax.set_xlabel('Cycle')
plt.suptitle("Sensor degradation trajectories (sample engines)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('outputs/eda_degradation_trajectories.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved: eda_degradation_trajectories.png ✓")

# ============================================================
# 6. SENSOR DISTRIBUTIONS
# ============================================================
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for ax, col in zip(axes.flatten(), RAW_FEATURES):
    ax.hist(train_df[col], bins=50, color='steelblue', alpha=0.75, edgecolor='none')
    ax.set_title(col, fontsize=9)
    ax.set_yticks([])
for ax in axes.flatten()[len(RAW_FEATURES):]:
    ax.set_visible(False)
plt.suptitle("Sensor distributions (raw, train set)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('outputs/eda_distributions.png', dpi=130, bbox_inches='tight')
plt.show()

# ============================================================
# 7. EWMA ROLLING FEATURES  (captures degradation TREND)
# ============================================================
ALPHA = 0.1   # smoothing factor — lower = longer memory

def add_ewma_features(df, sensor_cols, alpha=ALPHA):
    """Add per-unit EWMA for each sensor column."""
    df = df.copy().sort_values(['unit', 'cycle'])
    for col in sensor_cols:
        ewma_col = f'{col}_ewma'
        df[ewma_col] = (
            df.groupby('unit')[col]
              .transform(lambda x: x.ewm(alpha=alpha, adjust=False).mean())
        )
    return df

train_df = add_ewma_features(train_df, RAW_FEATURES)
test_df  = add_ewma_features(test_df,  RAW_FEATURES)

EWMA_FEATURES = [f'{c}_ewma' for c in RAW_FEATURES]
ALL_FEATURES  = RAW_FEATURES + EWMA_FEATURES
print(f"\nTotal features after EWMA: {len(ALL_FEATURES)}")

# ============================================================
# 8. GLOBAL NORMALIZATION  (fit on train only → apply to both)
#    FIX: original code used per-unit scaler on train + global on test
#         → distribution mismatch that artificially inflated train metrics
# ============================================================
scaler = MinMaxScaler()
train_df[ALL_FEATURES] = scaler.fit_transform(train_df[ALL_FEATURES])
test_df[ALL_FEATURES]  = scaler.transform(test_df[ALL_FEATURES])
print("Global normalization done ✓")

# ============================================================
# 9. VIF — multicollinearity check  (on raw sensor features only)
# ============================================================
X_vif = train_df[RAW_FEATURES].values
vif_df = pd.DataFrame({
    'feature': RAW_FEATURES,
    'VIF'    : [variance_inflation_factor(X_vif, i)
                for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)

print("\nVIF scores (raw sensor features):")
print(vif_df.to_string(index=False))
high_vif = vif_df[vif_df['VIF'] > 10]['feature'].tolist()
print(f"\nHigh-VIF features (>10): {high_vif}")

# Save VIF plot
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato' if v > 10 else 'steelblue' for v in vif_df['VIF']]
ax.barh(vif_df['feature'], vif_df['VIF'], color=colors)
ax.axvline(10, color='red', linestyle='--', label='VIF=10 threshold')
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factors (multicollinearity check)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/vif_plot.png', dpi=130)
plt.show()

# ============================================================
# 10. PREPARE ARRAYS FOR MODELLING
#     GroupKFold by unit — prevents same-engine rows leaking across folds
# ============================================================
X_train_full = train_df[ALL_FEATURES].values
y_train_full = train_df['label'].values
groups       = train_df['unit'].values   # group labels for GroupKFold

X_test_full  = test_df[ALL_FEATURES].values
y_test_full  = test_df['label'].values

print(f"\nX_train_full: {X_train_full.shape}")
print(f"X_test_full : {X_test_full.shape}")

# ── Group-based fold (engine = group; FIX vs. random KFold) ─
gkf = GroupKFold(n_splits=5)

# ============================================================
# 11. LOGISTIC REGRESSION — tune C
# ============================================================
lr_pipe = Pipeline([
    ('clf', LogisticRegression(max_iter=3000, class_weight='balanced',
                               solver='lbfgs'))
])
lr_param_grid = {'clf__C': [0.001, 0.01, 0.1, 1, 10, 100]}

lr_search = GridSearchCV(lr_pipe, lr_param_grid, cv=gkf,
                         scoring='f1', n_jobs=-1, verbose=0)
lr_search.fit(X_train_full, y_train_full, groups=groups)
best_lr = lr_search.best_estimator_

print(f"\n[LR] Best C      : {lr_search.best_params_['clf__C']}")
print(f"[LR] Best CV F1  : {lr_search.best_score_:.3f}")

# ============================================================
# 12. GAUSSIAN NAIVE BAYES — tune var_smoothing
# ============================================================
gnb_pipe = Pipeline([('clf', GaussianNB())])
gnb_param_grid = {'clf__var_smoothing': np.logspace(-11, -7, 15)}

gnb_search = GridSearchCV(gnb_pipe, gnb_param_grid, cv=gkf,
                          scoring='f1', n_jobs=-1, verbose=0)
gnb_search.fit(X_train_full, y_train_full, groups=groups)
best_gnb = gnb_search.best_estimator_

print(f"\n[GNB] Best var_smoothing : {gnb_search.best_params_['clf__var_smoothing']:.2e}")
print(f"[GNB] Best CV F1         : {gnb_search.best_score_:.3f}")

# ============================================================
# 13. RANDOM FOREST — tune n_estimators, max_depth
# ============================================================
rf_pipe = Pipeline([
    ('clf', RandomForestClassifier(class_weight='balanced',
                                   random_state=42, n_jobs=-1))
])
rf_param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth'   : [8, 15, None],
    'clf__min_samples_leaf': [5, 20]
}

rf_search = GridSearchCV(rf_pipe, rf_param_grid, cv=gkf,
                         scoring='f1', n_jobs=-1, verbose=0)
rf_search.fit(X_train_full, y_train_full, groups=groups)
best_rf = rf_search.best_estimator_

print(f"\n[RF] Best params : {rf_search.best_params_}")
print(f"[RF] Best CV F1  : {rf_search.best_score_:.3f}")

# ============================================================
# 14. 5-FOLD GROUP CROSS-VALIDATION (GroupKFold on all models)
# ============================================================
scoring = ['f1', 'roc_auc', 'precision', 'recall']

lr_cv  = cross_validate(best_lr,  X_train_full, y_train_full,
                        cv=gkf, groups=groups,
                        scoring=scoring, return_train_score=True)
gnb_cv = cross_validate(best_gnb, X_train_full, y_train_full,
                        cv=gkf, groups=groups,
                        scoring=scoring, return_train_score=True)
rf_cv  = cross_validate(best_rf,  X_train_full, y_train_full,
                        cv=gkf, groups=groups,
                        scoring=scoring, return_train_score=True)

def print_cv_summary(name, cv_scores):
    f1s       = cv_scores['test_f1']
    tr_f1s    = cv_scores['train_f1']
    mean, std = f1s.mean(), f1s.std()
    ci        = 1.96 * std / np.sqrt(len(f1s))
    gap       = tr_f1s.mean() - mean       # train-val gap (overfit indicator)
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"  Val  F1 per fold : {np.round(f1s, 3)}")
    print(f"  Train F1 (mean)  : {tr_f1s.mean():.3f}   |  Val F1: {mean:.3f} ± {std:.3f}")
    print(f"  Train-Val gap    : {gap:.3f}  {'⚠ overfit' if gap > 0.15 else '✓ OK'}")
    print(f"  95% CI           : [{mean-ci:.3f},  {mean+ci:.3f}]")
    print(f"  AUC              : {cv_scores['test_roc_auc'].mean():.3f}")
    print(f"  Precision        : {cv_scores['test_precision'].mean():.3f}")
    print(f"  Recall           : {cv_scores['test_recall'].mean():.3f}")

print_cv_summary("Logistic Regression", lr_cv)
print_cv_summary("Gaussian Naive Bayes", gnb_cv)
print_cv_summary("Random Forest",       rf_cv)

# ============================================================
# 15. TEST SET EVALUATION  (uses RUL_FD001.txt ground truth)
#     FIX: original code NEVER evaluated on the held-out test set
# ============================================================
print("\n" + "="*50)
print("  HELD-OUT TEST SET EVALUATION")
print("="*50)

models_eval = [
    ("Logistic Regression", best_lr),
    ("Gaussian Naive Bayes", best_gnb),
    ("Random Forest",        best_rf),
]

test_results = {}
for name, model in models_eval:
    y_pred  = model.predict(X_test_full)
    y_prob  = model.predict_proba(X_test_full)[:, 1]
    results = {
        'F1'       : f1_score(y_test_full, y_pred),
        'AUC'      : roc_auc_score(y_test_full, y_prob),
        'Precision': precision_score(y_test_full, y_pred),
        'Recall'   : recall_score(y_test_full, y_pred),
    }
    test_results[name] = results
    print(f"\n  {name}")
    for metric, val in results.items():
        print(f"    {metric:10s}: {val:.3f}")
    print(f"\n  Detailed report:\n{classification_report(y_test_full, y_pred, target_names=['Normal','Near-failure'])}")

# ============================================================
# 16. PLOT — ROC CURVES (Validation + Test side by side)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Val-set ROC (using last GroupKFold split for display)
train_idx, val_idx = list(gkf.split(X_train_full, y_train_full, groups))[4]
X_val_gk, y_val_gk = X_train_full[val_idx], y_train_full[val_idx]

ax = axes[0]
for name, model in models_eval:
    RocCurveDisplay.from_estimator(model, X_val_gk, y_val_gk, ax=ax, name=name)
ax.plot([0,1],[0,1], 'k--', lw=0.8, label='Random')
ax.set_title('ROC — Cross-Val Fold 5')
ax.legend(fontsize=8)

# Test-set ROC
ax = axes[1]
for name, model in models_eval:
    RocCurveDisplay.from_estimator(model, X_test_full, y_test_full, ax=ax, name=name)
ax.plot([0,1],[0,1], 'k--', lw=0.8, label='Random')
ax.set_title('ROC — Held-Out Test Set (RUL_FD001.txt)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('outputs/roc_curves.png', dpi=150)
plt.show()
print("Saved: roc_curves.png ✓")

# ============================================================
# 17. PLOT — CONFUSION MATRICES (Test Set)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models_eval):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test_full, y_test_full, ax=ax,
        display_labels=['Normal', 'Near-failure'],
        colorbar=False, cmap='Blues'
    )
    ax.set_title(f'{name}\n(Test Set)')
plt.suptitle('Confusion Matrices — Held-Out Test Set', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('outputs/confusion_matrices_test.png', dpi=150)
plt.show()
print("Saved: confusion_matrices_test.png ✓")

# ============================================================
# 18. PLOT — LEARNING CURVES
# ============================================================
def plot_lc(model, X, y, groups, name, ax):
    # Use GroupKFold in learning curve to avoid leakage
    sizes, tr_sc, val_sc = learning_curve(
        model, X, y, cv=GroupKFold(n_splits=5),
        groups=groups,
        scoring='f1',
        train_sizes=np.linspace(0.2, 1.0, 8),
        n_jobs=-1
    )
    ax.plot(sizes, tr_sc.mean(1),  label='Train F1', color='steelblue')
    ax.fill_between(sizes,
                    tr_sc.mean(1) - tr_sc.std(1),
                    tr_sc.mean(1) + tr_sc.std(1),
                    alpha=0.15, color='steelblue')
    ax.plot(sizes, val_sc.mean(1), label='Val F1',   color='coral')
    ax.fill_between(sizes,
                    val_sc.mean(1) - val_sc.std(1),
                    val_sc.mean(1) + val_sc.std(1),
                    alpha=0.15, color='coral')
    gap = tr_sc.mean(1)[-1] - val_sc.mean(1)[-1]
    ax.set_title(f'Learning Curve — {name}\n(Train–Val gap: {gap:.2f})')
    ax.set_xlabel('Training samples')
    ax.set_ylabel('F1')
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, models_eval):
    plot_lc(model, X_train_full, y_train_full, groups, name, ax)
plt.tight_layout()
plt.savefig('outputs/learning_curves.png', dpi=150)
plt.show()
print("Saved: learning_curves.png ✓")

# ============================================================
# 19. PLOT — FEATURE IMPORTANCES (Random Forest)
# ============================================================
importances = best_rf.named_steps['clf'].feature_importances_
feat_imp_df = (
    pd.DataFrame({'feature': ALL_FEATURES, 'importance': importances})
    .sort_values('importance', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['tomato' if '_ewma' in f else 'steelblue' for f in feat_imp_df['feature']]
ax.barh(feat_imp_df['feature'][::-1], feat_imp_df['importance'][::-1], color=colors[::-1])
ax.set_xlabel('Gini Importance')
ax.set_title('Top 20 Feature Importances — Random Forest\n(red = EWMA trend, blue = raw sensor)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='tomato', label='EWMA trend feature'),
                   Patch(color='steelblue', label='Raw sensor feature')],
          loc='lower right')
plt.tight_layout()
plt.savefig('outputs/feature_importances.png', dpi=150)
plt.show()
print("Saved: feature_importances.png ✓")

# ============================================================
# 20. SUMMARY TABLE
# ============================================================
summary_rows = []
cv_lookup = {'Logistic Regression': lr_cv,
             'Gaussian Naive Bayes': gnb_cv,
             'Random Forest': rf_cv}

for name, model in models_eval:
    cv = cv_lookup[name]
    tr = test_results[name]
    summary_rows.append({
        'Model'        : name,
        'CV F1 (mean)' : f"{cv['test_f1'].mean():.3f}",
        'CV AUC'       : f"{cv['test_roc_auc'].mean():.3f}",
        'Train-Val Gap': f"{cv['train_f1'].mean()-cv['test_f1'].mean():.3f}",
        'Test F1'      : f"{tr['F1']:.3f}",
        'Test AUC'     : f"{tr['AUC']:.3f}",
        'Test Precision': f"{tr['Precision']:.3f}",
        'Test Recall'  : f"{tr['Recall']:.3f}",
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*70)
print("  FINAL SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))

# ============================================================
# 21. EXPORT & SAVE
# ============================================================
train_df.to_csv('train_clean.csv', index=False)
test_df.to_csv('test_clean.csv',   index=False)

joblib.dump(best_lr,      'models/lr_model.pkl')
joblib.dump(best_gnb,     'models/gnb_model.pkl')
joblib.dump(best_rf,      'models/rf_model.pkl')
joblib.dump(ALL_FEATURES, 'models/feature_cols.pkl')
joblib.dump(scaler,       'models/global_scaler.pkl')

print("\nExported CSVs and models ✓")
print("\n✅ Improved pipeline complete!")